# NLP Masterclass 02: Word Embeddings & Word2Vec

**Track:** Natural Language Processing (0 to Masterclass)  
**Prerequisites:** NLP 01 (Tokenization), ML Fundamentals  
**Difficulty:** ⭐⭐ Intermediate  
**Estimated Time:** 120–150 minutes

---


## 🎓 Socratic Deep Check

> *"J.R. Firth famously said, 'You shall know a word by the company it keeps.' If every word's meaning is defined by its neighbors, then synonymity becomes a geometric property. But Word2Vec gives 'bank' the same vector regardless of context. How does this 'static' limitation motivate the jump to Transformers, and why do we still use Word2Vec for specialized tasks today?"*

---

## The Core Problem: How Machines "Read"

Computers are calculation engines for linear algebra, not linguists. For a model to process language, we must map discrete tokens into a continuous vector space $\mathbb{R}^d$. This notebook explores the **Representation Learning** revolution that changed NLP forever: the shift from sparse, high-dimensional counts to dense, low-dimensional embeddings.


## 📑 Table of Contents

* [🎓 Socratic Deep Check](#socratic-deep-check)
* [The Core Problem: How Machines "Read"](#the-core-problem-how-machines-read)
* [1 · The Distributional Hypothesis](#1-the-distributional-hypothesis)
  * [Localist vs. Distributed Representations](#localist-vs-distributed-representations)
  * [The Distributional Hypothesis](#the-distributional-hypothesis)
* [2 · The Word2Vec Revolution](#2-the-word2vec-revolution)
  * [A. Continuous Bag-of-Words (CBOW)](#a-continuous-bag-of-words-cbow)
  * [B. Skip-Gram](#b-skip-gram)
  * [The Mathematical Objective](#the-mathematical-objective)
* [3 · Training Acceleration: Two Solutions to the Softmax Bottleneck](#3-training-acceleration-two-solutions-to-the-softmax-bottleneck)
  * [3A · Negative Sampling (SGNS)](#3a-negative-sampling-sgns)
  * [3B · Hierarchical Softmax](#3b-hierarchical-softmax)
    * [The Core Idea](#the-core-idea)
    * [Probability of a Word](#probability-of-a-word)
    * [Complexity Reduction](#complexity-reduction)
    * [When to Use Which?](#when-to-use-which)
  * [3C · Sub-sampling of Frequent Words](#3c-sub-sampling-of-frequent-words)
    * [The Mathematical Intuition](#the-mathematical-intuition)
    * [The Mikolov Sub-sampling Formula](#the-mikolov-sub-sampling-formula)
* [4 · Skip-Gram with Negative Sampling (PyTorch from Scratch)](#4-skip-gram-with-negative-sampling-pytorch-from-scratch)
  * [# TEST IT: Verification](#test-it-verification)
* [5 · GloVe: Global Vectors](#5-glove-global-vectors)
  * [The Key Difference](#the-key-difference)
  * [The GloVe Objective](#the-glove-objective)
* [6 · FastText: Handling Subwords & OOV](#6-fasttext-handling-subwords-oov)
  * [Example: "where" with n=3](#example-where-with-n3)
* [7 · Intrinsic Evaluation](#7-intrinsic-evaluation)
  * [1. Word Similarity](#1-word-similarity)
  * [2. Analogies (Vector Arithmetic)](#2-analogies-vector-arithmetic)
  * [Limitations of Intrinsic Evaluation](#limitations-of-intrinsic-evaluation)
* [8 · Pushing Static Embeddings to the Limit: Phrases and Polysemy](#8-pushing-static-embeddings-to-the-limit-phrases-and-polysemy)
  * [🎓 Socratic Deep Check](#socratic-deep-check)
  * [8.1 · Phrase Merging via PMI](#81-phrase-merging-via-pmi)
  * [8.2 · Sense2Vec: Solving the Polysemy Problem](#82-sense2vec-solving-the-polysemy-problem)
* [9 · The Bridge to Context: ELMo (Embeddings from Language Models)](#9-the-bridge-to-context-elmo-embeddings-from-language-models)
  * [🎓 Socratic Probe](#socratic-probe)
  * [The Architecture of ELMo (2018)](#the-architecture-of-elmo-2018)
  * [Why ELMo Changed Everything](#why-elmo-changed-everything)
* [10 · Hyperbolic (Poincaré) Embeddings](#10-hyperbolic-poincaré-embeddings)
  * [🎓 Socratic Deep Check](#socratic-deep-check)
  * [The Flaw of Euclidean Space](#the-flaw-of-euclidean-space)
  * [The Poincaré Ball Model](#the-poincaré-ball-model)
* [11 · Cross-Lingual Alignment (MUSE)](#11-cross-lingual-alignment-muse)
  * [🎓 Socratic Deep Check](#socratic-deep-check)
  * [The Intuition: Language Isomorphism](#the-intuition-language-isomorphism)
  * [The Mathematical Magic: Orthogonal Procrustes](#the-mathematical-magic-orthogonal-procrustes)
  * [Embedding Space Analysis](#embedding-space-analysis)
* [✅ Knowledge Check](#knowledge-check)
  * [Q1: The Softmax Bottleneck](#q1-the-softmax-bottleneck)
  * [Q2: Hierarchical Softmax Probability Guarantee](#q2-hierarchical-softmax-probability-guarantee)
  * [Q3: FastText vs. Word2Vec](#q3-fasttext-vs-word2vec)
  * [Q4: Frozen vs. Fine-Tuned Embeddings](#q4-frozen-vs-fine-tuned-embeddings)
  * [Q5: When to Use Static Embeddings?](#q5-when-to-use-static-embeddings)
* [🔨 Practical Practice](#practical-practice)
* [12 · Embedding Bias, Fairness, and WEAT](#12-embedding-bias-fairness-and-weat)
  * [Mitigation: Hard Debiasing](#mitigation-hard-debiasing)


In [ ]:
!pip install -q torch numpy matplotlib scikit-learn gensim tqdm

## 1 · The Distributional Hypothesis

### Localist vs. Distributed Representations

In a **Localist** representation (like One-Hot Encoding), each word is an independent dimension. 
- **Curse of Dimensionality**: A 100k-word vocabulary requires 100k-dimensional vectors.
- **Orthogonality**: All words are equidistant. `cosine(cat, dog) = 0`, the same as `cosine(cat, refrigerator)`.

A **Distributed** representation (Embedding) represents words as dense vectors where meaning is "distributed" across multiple dimensions.

### The Distributional Hypothesis
The bedrock of modern NLP is the **Distributional Hypothesis**: Words that occur in similar contexts tend to have similar meanings. 

> "A word is characterized by the company it keeps." — J.R. Firth (1957)

If we can predict a word's context, we can "learn" its meaning implicitly. This is the foundation of **Self-Supervised Learning**.

--- 
## 2 · The Word2Vec Revolution

In 2013, Mikolov et al. introduced two architectures that efficiently learned word vectors from massive datasets.

### A. Continuous Bag-of-Words (CBOW)
**Task**: Predict the *center* word given the surrounding *context* words.
- Example: `"The [?] sat on the mat"` $\rightarrow$ predict `"cat"`.
- Good for frequent words; faster to train.

### B. Skip-Gram
**Task**: Predict the *context* words given the *center* word.
- Example: `"cat"` $\rightarrow$ predict `["The", "sat", "on"]`.
- Better at representing rare words and capturing complex relationships.

### The Mathematical Objective
The standard objective function for Skip-Gram is to maximize the log-likelihood of the context words $w_{t+j}$ given the center word $w_t$:

$$J(\theta) = \frac{1}{T} \sum_{t=1}^T \sum_{-m \le j \le m, j \neq 0} \log P(w_{t+j} | w_t)$$

The probability is typically modeled via **Softmax**:

$$P(o | c) = \frac{\exp(u_o^T v_c)}{\sum_{w=1}^V \exp(u_w^T v_c)}$$

**The Problem:** The denominator requires summing over the entire vocabulary $V$ (e.g., 500,000 words), making it computationally impossible for large datasets.

--- 
## 3 · Training Acceleration: Two Solutions to the Softmax Bottleneck

The full softmax denominator $\sum_{w=1}^V \exp(u_w^T v_c)$ has $O(V)$ complexity per training sample. Two landmark techniques reduce this to tractable levels.

### 3A · Negative Sampling (SGNS)

Instead of predicting the exact word (Multi-class Classification), we turn it into a **Binary Classification** task:
1. **Positive Pair**: (center_word, actual_context_word) $\rightarrow$ Predict 1
2. **Negative Pairs**: (center_word, random_noise_word) $\rightarrow$ Predict 0

The new objective function becomes:

$$J_{SGNS} = \log \sigma(u_o^T v_c) + \sum_{i=1}^k \mathbb{E}_{w_i \sim P_n(w)} [\log \sigma(-u_i^T v_c)]$$

where $\sigma$ is the sigmoid function and $k$ is the number of negative samples.

**Complexity:** $O(k)$ where $k \ll V$ (typically $k \in [5, 20]$).

**Noise Distribution:** Negative words are sampled from a smoothed unigram distribution $P_n(w) \propto \text{count}(w)^{3/4}$. The $3/4$ exponent upweights rare words relative to pure frequency, giving them more chances to appear as negatives and improving the quality of the learned space.

### 3B · Hierarchical Softmax

Hierarchical Softmax is the other major technique from the original Word2Vec paper. Instead of computing a flat softmax over $V$ words, it replaces the output layer with a **binary Huffman tree** where each leaf is a word and each internal node has a learned weight vector.

#### The Core Idea

To predict word $w$, we no longer compute one giant softmax. Instead, we walk a **path from the root to the leaf** $w$ in the binary tree, making a sequence of binary (left/right) decisions at each internal node.

```
                    [root θ₁]
                   /          \
              [θ₂]            [θ₃]
             /    \          /    \
          [θ₄]   [θ₅]    "cat"  "sat"
         /    \
      "the"  "dog"
```

Each internal node $n$ at depth $d$ along the path to word $w$ makes a binary decision using its own parameter vector $\theta_n$:

$$P(\text{go left at node } n \mid v_c) = \sigma(\theta_n^T v_c)$$
$$P(\text{go right at node } n \mid v_c) = 1 - \sigma(\theta_n^T v_c)$$

#### Probability of a Word

The probability of predicting word $w$ is the **product of all binary decisions** along its unique path from root to leaf:

$$P(w \mid v_c) = \prod_{j=1}^{L(w)} \sigma\bigl(\llbracket n(w,j+1) = \text{left}(n(w,j)) \rrbracket \cdot \theta_{n(w,j)}^T v_c\bigr)$$

where:
- $L(w)$ is the **path length** (number of internal nodes from root to leaf $w$),
- $n(w, j)$ is the $j$-th node on the path to word $w$,
- $\llbracket \cdot \rrbracket$ is $+1$ if the next step is left, $-1$ if right.

#### Complexity Reduction

| Approach | Complexity per sample | Key Operation |
|---|---|---|
| Full Softmax | $O(V)$ | Dot product with every word vector |
| Negative Sampling | $O(k)$ | Binary classification vs. $k$ noise words |
| Hierarchical Softmax | $O(\log V)$ | Binary decisions along tree path |

Using a **Huffman tree** (frequent words get shorter paths) further optimizes this — the most common words require fewer binary decisions, concentrating computation where it matters most.

#### When to Use Which?

- **Negative Sampling** is preferred for **large corpora** and when **rare word accuracy** is important (the standard in production).
- **Hierarchical Softmax** works well for **smaller datasets** and vocabularies with highly **skewed frequency distributions**, where the Huffman encoding gives significant speedup.

> 🎓 **Socratic Probe**: *Hierarchical Softmax guarantees that $\sum_{w=1}^V P(w | v_c) = 1$ — it's a valid probability distribution. Why does Negative Sampling NOT guarantee this property, and why doesn't it matter in practice?*

### 3C · Sub-sampling of Frequent Words

While Negative Sampling and Hierarchical Softmax accelerate the *output* layer, **Sub-sampling** optimizes the *input* data itself.

#### The Mathematical Intuition
In any large corpus, words like "the", "a", and "is" appear millions of times but carry very little semantic information. Conversely, a rare word like "quantum" appears infrequently but is highly informative.
- **Redundancy**: Seeing the pair `(the, quick)` for the 1,000,000th time provides almost no new gradient signal to the model.
- **Efficiency**: If we can avoid processing these redundant pairs, we can train on much larger corpora in the same amount of time.

#### The Mikolov Sub-sampling Formula
To counter this imbalance, Word2Vec discards words from the training stream with a probability $P(w_i)$ based on their frequency $f(w_i)$:

$$P(w_i) = 1 - \sqrt{\frac{t}{f(w_i)}}$$

Where:
- $f(w_i)$ is the fraction of total words in the corpus that are $w_i$.
- $t$ is a threshold parameter (typically $10^{-5}$ or $10^{-3}$).

**How it works:**
- If $f(w_i) \le t$ (rare words), the probability is $\le 0$, so the word is **never** dropped.
- If $f(w_i) > t$ (frequent words), the probability of dropping increases with frequency.
- **Result:** This effectively "flattens" the word distribution, giving the model more "quality time" with rare, informative words while skipping the noise of stop-words.


--- 
## 4 · Skip-Gram with Negative Sampling (PyTorch from Scratch)

We will implement a minimal but production-quality version of SGNS to understand the mechanics of embedding updates.

In [ ]:
from __future__ import annotations

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
from collections import Counter
from typing import List, Dict, Tuple

# ──────────────────────────── Hyperparameters ────────────────────────────
EMBED_DIM: int = 16
WINDOW_SIZE: int = 2
N_NEGATIVE_SAMPLES: int = 5
BATCH_SIZE: int = 8
LEARNING_RATE: float = 0.01
N_EPOCHS: int = 100
SMOOTHING_EXPONENT: float = 0.75  # Unigram distribution smoothing (Mikolov et al.)
SUBSAMPLING_THRESHOLD: float = 1e-5

# ──────────────────────────── 1. Corpus & Vocabulary ─────────────────────
text: str = """The quick brown fox jumps over the lazy dog. 
          Data science is an interdisciplinary field that uses scientific methods, 
          processes, algorithms and systems to extract knowledge and insights 
          from noisy, structured and unstructured data."""

tokens: List[str] = text.lower().replace('.', '').replace(',', '').split()
vocab: Dict[str, int] = {w: i for i, w in enumerate(sorted(set(tokens)))}
idx_to_word: Dict[int, str] = {i: w for w, i in vocab.items()}
VOCAB_SIZE: int = len(vocab)
word_counts: Counter = Counter(tokens)

# Negative Sampling Distribution: P(w)^(3/4) as per the original paper.
# This smoothing upweights rare words relative to pure unigram frequency.
freqs: np.ndarray = np.array([word_counts[idx_to_word[i]] for i in range(VOCAB_SIZE)])
unigram_dist: np.ndarray = freqs ** SMOOTHING_EXPONENT / np.sum(freqs ** SMOOTHING_EXPONENT)
UNIGRAM_TENSOR: torch.Tensor = torch.from_numpy(unigram_dist).float()

print(f"Corpus tokens: {len(tokens)} | Vocabulary size: {VOCAB_SIZE}")
print(f"Sample vocab entries: {list(vocab.items())[:5]}")

In [ ]:
# ──────────────────────────── 2. Dataset ─────────────────────────────────

class SGNSDataset(Dataset):
    """Generates (center, context, negatives) triplets for Skip-Gram training.
    
    For each token in the corpus, creates pairs with every word within
    the specified window. Negative samples are drawn lazily at __getitem__
    time from the smoothed unigram distribution.
    """

    def __init__(
        self,
        tokens: List[str],
        vocab: Dict[str, int],
        word_counts: Counter,
        window_size: int = WINDOW_SIZE,
        n_negatives: int = N_NEGATIVE_SAMPLES,
        subsampling_threshold: float = SUBSAMPLING_THRESHOLD,
    ) -> None:
        self.n_negatives = n_negatives

        # 1. Apply Sub-sampling Filter (Mikolov et al.)
        total_words = sum(word_counts.values())
        filtered_tokens = []
        for token in tokens:
            freq = word_counts[token] / total_words
            if freq > subsampling_threshold:
                # Mikolov formula probability of dropping
                p_drop = 1 - np.sqrt(subsampling_threshold / freq)
                if np.random.random() < p_drop:
                    continue # Drop the word
            filtered_tokens.append(token)

        # 2. Generate Pairs from Filtered Stream
        self.pairs: List[Tuple[int, int]] = []
        for i, token in enumerate(filtered_tokens):
            center_idx: int = vocab[token]
            window_start: int = max(0, i - window_size)
            window_end: int = min(len(filtered_tokens), i + window_size + 1)
            for j in range(window_start, window_end):
                if i != j:
                    self.pairs.append((center_idx, vocab[filtered_tokens[j]]))
    def __len__(self) -> int:
        return len(self.pairs)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        center, context = self.pairs[idx]
        negatives: torch.Tensor = torch.multinomial(
            UNIGRAM_TENSOR, self.n_negatives, replacement=True
        )
        return torch.tensor(center), torch.tensor(context), negatives


dataset = SGNSDataset(tokens, vocab)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
print(f"Total training pairs: {len(dataset)}")

In [ ]:
# ──────────────────────────── 3. Model ───────────────────────────────────

class SGNSModel(nn.Module):
    """Skip-Gram with Negative Sampling.

    Maintains two separate embedding matrices:
      - `in_embed`  (W):  input / center-word embeddings
      - `out_embed` (W'): output / context-word embeddings

    After training, `in_embed.weight` is used as the final word vectors.
    """

    def __init__(self, vocab_size: int, embed_dim: int) -> None:
        super().__init__()
        self.in_embed = nn.Embedding(vocab_size, embed_dim)
        self.out_embed = nn.Embedding(vocab_size, embed_dim)

        # Xavier-style initialization for input, zero for output (Mikolov convention)
        nn.init.uniform_(self.in_embed.weight, -0.5 / embed_dim, 0.5 / embed_dim)
        nn.init.zeros_(self.out_embed.weight)

    def forward(
        self,
        center: torch.Tensor,    # (B,)
        context: torch.Tensor,   # (B,)
        negatives: torch.Tensor, # (B, k)
    ) -> torch.Tensor:
        """Compute the SGNS loss.

        Args:
            center:    Indices of center words.             Shape (B,)
            context:   Indices of true context words.       Shape (B,)
            negatives: Indices of negative-sampled words.   Shape (B, k)

        Returns:
            Scalar loss averaged over the batch.
        """
        EPS: float = 1e-10

        v_c: torch.Tensor = self.in_embed(center)     # (B, D)
        u_o: torch.Tensor = self.out_embed(context)   # (B, D)
        u_n: torch.Tensor = self.out_embed(negatives) # (B, k, D)

        # ── Positive loss: maximize σ(u_o · v_c) ──
        pos_score: torch.Tensor = torch.sum(v_c * u_o, dim=1)             # (B,)
        pos_loss: torch.Tensor = -torch.log(torch.sigmoid(pos_score) + EPS).mean()

        # ── Negative loss: maximize σ(-u_n · v_c) ──
        # (B, 1, D) @ (B, D, k) → (B, 1, k) → squeeze → (B, k)
        neg_score: torch.Tensor = torch.bmm(u_n, v_c.unsqueeze(2)).squeeze(2)
        neg_loss: torch.Tensor = -torch.log(torch.sigmoid(-neg_score) + EPS).sum(1).mean()

        return pos_loss + neg_loss

In [ ]:
# ──────────────────────────── 4. Training Loop ───────────────────────────

model = SGNSModel(VOCAB_SIZE, EMBED_DIM)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print("Training SGNS...")
for epoch in range(1, N_EPOCHS + 1):
    epoch_loss: float = 0.0
    for center, context, negs in dataloader:
        optimizer.zero_grad()
        loss: torch.Tensor = model(center, context, negs)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    if epoch % 20 == 0:
        avg_loss: float = epoch_loss / len(dataloader)
        print(f"  Epoch {epoch:3d}/{N_EPOCHS} | Loss: {avg_loss:.4f}")

print("✓ Training complete.")

### # TEST IT: Verification
Let's see if semantically related words are close in our tiny vector space.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity


def get_similarity(w1: str, w2: str) -> float:
    """Compute cosine similarity between two words using learned embeddings."""
    v1: np.ndarray = model.in_embed.weight[vocab[w1]].detach().numpy()
    v2: np.ndarray = model.in_embed.weight[vocab[w2]].detach().numpy()
    return float(cosine_similarity([v1], [v2])[0][0])


# Words that co-occur in the "data science" paragraph should be closer
# than words from unrelated sentences.
print(f"Similarity (data, knowledge): {get_similarity('data', 'knowledge'):.3f}")
print(f"Similarity (data, fox):       {get_similarity('data', 'fox'):.3f}")
print(f"Similarity (quick, brown):    {get_similarity('quick', 'brown'):.3f}")

--- 
## 5 · GloVe: Global Vectors

While Word2Vec is a **predictive** model (iterates over local windows), **GloVe** (Pennington et al., 2014) is a **count-based matrix factorization** model.

### The Key Difference
- **Word2Vec Strategy**: Scan text window by window. Fails to leverage global statistics directly.
- **GloVe Strategy**: Build a global **Co-occurrence Matrix** $X$ first. Then factorize it to get embeddings.

### The GloVe Objective
GloVe minimizes the difference between the dot product of two vectors and the log of their co-occurrence frequency, weighted by a function $f(X_{ij})$ to prevent common words from dominating:

$$J = \sum_{i,j=1}^V f(X_{ij}) (w_i^T \tilde{w}_j + b_i + \tilde{b}_j - \log X_{ij})^2$$

**Why use GloVe?**
- Better at capturing global semantic relationships.
- Often more stable for smaller datasets than Word2Vec.

--- 
## 6 · FastText: Handling Subwords & OOV

Word2Vec and GloVe treat words as atomic units. If a word wasn't in the training set (e.g., "antigravity"), they have no vector for it. This is the **Out-of-Vocabulary (OOV)** problem.

**FastText** (Bojanowski et al., 2016) solves this by treating each word as a bag of **character n-grams**.

### Example: "where" with n=3
- Subwords: `<wh`, `whe`, `her`, `ere`, `re>` (plus the word itself `<where>`)
- The vector for "where" is the **sum** of these subword vectors.

**Advantages:**
1. **OOV Support**: Even if the model hasn't seen "antigravity", it has seen "anti-" and "-gravity". It can construct a reasonable vector by summing subwords.
2. **Morphology**: It naturally captures relationships between "eat", "eats", and "eating" because they share character n-grams.

In [ ]:
from gensim.models import FastText

# Create a small toy dataset
sentences: list[list[str]] = [
    ["the", "speed", "of", "light", "is", "constant"], 
    ["gravity", "bends", "the", "fabric", "of", "space"],
]

# Train FastText
ft_model = FastText(sentences, vector_size=10, window=3, min_count=1, epochs=10)

# TEST OOV: "spacetime" was NOT in the training data
try:
    vector: np.ndarray = ft_model.wv['spacetime']
    print("✓ Successfully generated vector for OOV word 'spacetime'!")
    print(f"  Vector (first 5 dims): {vector[:5]}")
except KeyError:
    print("✗ FastText failed (this shouldn't happen!)")

--- 
## 7 · Intrinsic Evaluation

How do we know if our embeddings are "good"? **Intrinsic evaluation** measures the quality of the vector space *in isolation*, without any downstream task.

### 1. Word Similarity
Compare the model's similarity score with human-labeled datasets like **Wordsim-353**. If humans rate (cat, dog) at 9/10 and our cosine similarity is high, the model is learning well.

### 2. Analogies (Vector Arithmetic)
The most famous test of vector spaces is solving analogies: $A \text{ is to } B \text{ as } C \text{ is to } X$.

$$\text{vec(king)} - \text{vec(man)} + \text{vec(woman)} \approx \text{vec(queen)}$$

This works because the vector space encodes semantic **relations** as geometric **offsets**.

### Limitations of Intrinsic Evaluation

Intrinsic metrics can be **misleading**. A model scoring perfectly on word analogies may still perform poorly on sentiment analysis or named entity recognition. The embedding space captures the *right* relationships for analogy tasks but may miss the *discriminative features* needed for a specific downstream task.

This motivates the need for **extrinsic evaluation**: measuring embeddings by how well they perform on the task you actually care about.

---
## 8 · Pushing Static Embeddings to the Limit: Phrases and Polysemy

### 🎓 Socratic Deep Check
> *"If 'New' and 'York' are two separate vectors, how does Word2Vec understand the concept of a city? If 'apple' is always the same vector, how does it distinguish between a snack and a tech giant? Before we invented complex contextual models (ELMo/BERT), how did we 'hack' static embeddings to handle these fundamental linguistic challenges?"*

### 8.1 · Phrase Merging via PMI

Word2Vec, in its raw form, is a **unigram** model. It treats every token as an independent entity. This fails for multi-word expressions (MWEs) like *"New York"*, *"Ice Cream"*, or *"Machine Learning"*, where the meaning of the phrase is greater than the sum of its parts.

**The Solution: Data-Driven Phrase Detection**
In the original Word2Vec paper, Mikolov et al. used a simple statistical approach to identify phrases *before* training the embeddings. This involves calculating the **Pointwise Mutual Information (PMI)** (or a similar score) between adjacent words:

$$Score(w_i, w_j) = \frac{count(w_i w_j) - \delta}{count(w_i) \times count(w_j)}$$

Where $\delta$ is a discounting coefficient to prevent rare words from forming phrases. If the score exceeds a threshold, the words are fused into a single token (e.g., `New` + `York` → `New_York`). 

> **Engineering Benefit**: This allows the model to learn a specialized vector for the entity itself, drastically reducing the noise of "New" (the adjective) and "York" (the town in England) in the context of the American city.

### 8.2 · Sense2Vec: Solving the Polysemy Problem

The biggest weakness of static embeddings is **Polysemy** (one word, multiple meanings). Because there is a 1-to-1 mapping between a string and a vector, "apple" must represent both the fruit and the company simultaneously, resulting in a "blended" vector that is semantically suboptimal for both.

**The Sense2Vec Workaround (Trask et al., 2015)**
Before we had deep neural encoders to create "on-the-fly" contextual embeddings, **Sense2Vec** solved this by enriching the vocabulary with syntactic information. 

1. **Tagging**: Every word in the corpus is tagged with its Part-of-Speech (POS) or Entity Type.
2. **Augmentation**: The tag is appended to the word (e.g., `apple` → `apple|NOUN`, `apple` → `apple|PROPN`).
3. **Training**: Words with different tags are now treated as different tokens with separate vectors.

**Impact**: By using a simple, fast POS tagger as a pre-processor, we can force a static model to learn distinct semantic regions for different senses without the massive computational overhead of Transformers.


---
## 9 · The Bridge to Context: ELMo (Embeddings from Language Models)

### 🎓 Socratic Probe
> *Word2Vec and GloVe generate **static** embeddings—the word "bank" has the exact same vector in "river bank" and "bank account." Since we are about to study Recurrent Neural Networks (RNNs) which process sequences, how could we use their internal hidden states to make word vectors dynamic and context-aware?*

### The Architecture of ELMo (2018)
Before the Transformer took over, **ELMo** (Embeddings from Language Models) revolutionized NLP by introducing **contextualized word representations**. Unlike static models that look up a vector in a fixed table, ELMo computes embeddings on the fly using a deep, bi-directional LSTM.

1.  **Bi-directional Training**: ELMo trains two parallel LSTMs:
    *   **Forward LSTM**: Predicts the next word given the previous words (standard Language Modeling).
    *   **Backward LSTM**: Predicts the previous word given the future words.
2.  **Internal State Combination**: Instead of just using the final output, ELMo takes a linear combination of the **internal hidden states** from all layers of both LSTMs.
3.  **The Result**: The resulting vector for a word is a function of the entire input sentence. Mathematically, for a word at position $k$, the ELMo representation $E_k$ is:
    $$E_k = \gamma \sum_{j=0}^L s_j h_{k,j}$$
    where $h_{k,j}$ represents the concatenated hidden states of the $j$-th layer, and $s_j$ are task-specific weights learned during downstream fine-tuning.

### Why ELMo Changed Everything
ELMo proved that **context matters more than identity**. In 2018, simply swapping static Word2Vec vectors for ELMo embeddings yielded instant State-of-the-Art (SOTA) results across nearly every major NLP benchmark (SQuAD, NER, Sentiment). 

It established the "Pre-train then Fine-tune" paradigm that dominates AI today, setting the stage for the Transformer revolution by proving that a model's internal representations of language are its most valuable asset.

---
## 10 · Hyperbolic (Poincaré) Embeddings

### 🎓 Socratic Deep Check
> *"Euclidean space is 'flat'. When you move away from the origin, the volume grows polynomially. But consider WordNet—a massive hierarchy where each node has multiple children. As you go down the tree, the number of nodes grows exponentially. If you try to squash an exponential tree into a polynomial space, the leaves get crowded and the hierarchy collapses. How can we change the 'shape' of our vector space to respect the structural density of hierarchies?"*

### The Flaw of Euclidean Space
In high-dimensional Euclidean space, the distance between any two points $d(x, y) = ||x - y||_2$ follows standard geometry. However, in hierarchical data, the "true" relative distance between nodes behaves differently. To embed a tree structure faithfully, we need the distance to expand much more rapidly as we move away from the root.

### The Poincaré Ball Model
Hyperbolic space is a Riemannian manifold with constant negative curvature. The **Poincaré Ball** is a common model for this space: it uses an open $n$-dimensional ball where the distance between points increases exponentially as they approach the boundary (the edge of the ball).

- **The Intuition**: In a Poincaré Ball, the center is the "root". As you move toward the edges, the space "expands" because of the negative curvature. Lines that look straight are actually arcs.
- **Efficiency**: Because hyperbolic volume grows exponentially with radius, a relative small number of dimensions (e.g., $d=10$) can embed vast hierarchies (like the entire WordNet taxonomy) with almost zero distortion.
- **Impact**: Poincaré embeddings allow models to capture both the **similarity** (distance) and the **hierarchy** (radial position) simultaneously in a single vector space.


---
## 11 · Cross-Lingual Alignment (MUSE)

### 🎓 Socratic Deep Check
> *"Imagine you train 300-dimensional embeddings for English and 300-dimensional embeddings for Spanish. They have never seen each other. If 'gato' and 'cat' mean the same thing, will they have the same vector? Almost certainly not. But will 'gato' relate to 'perro' the same way 'cat' relates to 'dog'? If languages share a common semantic topological structure, can we simply rotate the Spanish space so it matches the English space?"*

### The Intuition: Language Isomorphism
Languages reflect a shared human experience. Regardless of the labels (words), the relationships between concepts tend to be **isomorphic** (similarly shaped). In 2017, the **MUSE** (Multilingual Unsupervised Word Embeddings) work by Conneau et al. proved that you can align these spaces without even using a dictionary.

### The Mathematical Magic: Orthogonal Procrustes
Suppose we have a set of source embeddings $X$ (English) and target embeddings $Y$ (Spanish). Our goal is to find a transformation matrix $W$ such that:

$$WX \approx Y$$

To preserve the quality of the embeddings (the angles and distances), $W$ must be an **orthogonal matrix** ($W^T W = I$). This is known as the **Orthogonal Procrustes problem**. 

**The Process:**
1.  **Independent Training**: Train English vectors and Spanish vectors separately on their respective monolingual corpora.
2.  **Domain Mapping**: Using a small adversarial signal or a few seed words, the model learns to "rotate" the Spanish space into the English space using $W$.
3.  **Result**: Without any parallel training data, the model can perform zero-shot translation. If you look at the nearest neighbor to the Spanish vector for *'manzana'* in the English space, it will be the vector for *'apple'*.

> **Masterclass Insight**: This alignment works because language-agnostic semantic structures are surprisingly robust. The "shape" of human thought, when projected into vector spaces, is universal regardless of the spoken tongue.


--- 
## 11 · Extrinsic Evaluation: Frozen vs. Fine-Tuned Embeddings

The ultimate test of any embedding is its performance on a **downstream task**. This section demonstrates a critical engineering decision:

| Strategy | `requires_grad` | Behavior |
|---|---|---|
| **Frozen** | `False` | Embeddings are fixed lookup tables. Only the classifier learns. |
| **Fine-Tuned** | `True` | Embeddings adapt to the task's loss signal. |

### Why Fine-Tuning Helps

Pre-trained embeddings encode *general* semantic relationships (king → queen, cat → dog). But for **sentiment analysis**, the model needs to learn that:
- "amazing" and "terrible" are **semantically similar** (both are strong adjectives → close in Word2Vec space)
- But they carry **opposite sentiment** (must be far apart for classification)

Fine-tuning allows the embedding space to **deform** to make task-relevant distinctions that the original training objective didn't capture.

> 🎓 **Socratic Probe**: *If fine-tuning always improves downstream accuracy, why not always fine-tune? What happens when your labeled dataset is very small (e.g., 50 examples)?*

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# EXTRINSIC EVALUATION: Sentiment Classification
# Frozen vs. Fine-Tuned Word2Vec Embeddings
# ──────────────────────────────────────────────────────────────────────────

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from typing import List, Tuple, Dict

# ──────────────────────────── 1. Synthetic Sentiment Dataset ──────────────
# We create a small but illustrative dataset where sentiment signal
# is carried by specific words. This lets us see the fine-tuning effect
# clearly without needing a large corpus.

SENTIMENT_DATA: List[Tuple[str, int]] = [
    # Positive (label = 1)
    ("this movie is amazing and wonderful", 1),
    ("absolutely brilliant film loved it", 1),
    ("great acting superb direction excellent", 1),
    ("good story beautiful cinematography", 1),
    ("fantastic performance really enjoyed", 1),
    ("wonderful experience highly recommended", 1),
    ("amazing story brilliant execution", 1),
    ("superb film excellent directing", 1),
    ("loved the beautiful music great", 1),
    ("really good absolutely fantastic", 1),
    # Negative (label = 0)
    ("this movie is terrible and boring", 0),
    ("absolutely awful film hated it", 0),
    ("bad acting poor direction horrible", 0),
    ("boring story terrible cinematography", 0),
    ("worst performance really disappointing", 0),
    ("horrible experience never again", 0),
    ("terrible story awful execution", 0),
    ("poor film horrible directing", 0),
    ("hated the bad music boring", 0),
    ("really bad absolutely awful", 0),
]


def build_vocab(data: List[Tuple[str, int]]) -> Tuple[Dict[str, int], int]:
    """Build vocabulary from sentiment dataset, reserving index 0 for <PAD>."""
    all_words: set = set()
    for text, _ in data:
        all_words.update(text.lower().split())
    word2idx: Dict[str, int] = {"<PAD>": 0}
    for i, w in enumerate(sorted(all_words), start=1):
        word2idx[w] = i
    return word2idx, len(word2idx)


def encode_sentence(
    text: str, word2idx: Dict[str, int], max_len: int
) -> torch.Tensor:
    """Convert a sentence to a padded tensor of word indices."""
    indices: List[int] = [
        word2idx.get(w, 0) for w in text.lower().split()
    ]
    # Pad or truncate to max_len
    indices = indices[:max_len] + [0] * max(0, max_len - len(indices))
    return torch.tensor(indices, dtype=torch.long)


# Build vocabulary and encode
sentiment_vocab, sentiment_vocab_size = build_vocab(SENTIMENT_DATA)
MAX_SEQ_LEN: int = 8

X: torch.Tensor = torch.stack(
    [encode_sentence(text, sentiment_vocab, MAX_SEQ_LEN) for text, _ in SENTIMENT_DATA]
)
y: torch.Tensor = torch.tensor([label for _, label in SENTIMENT_DATA], dtype=torch.float32)

print(f"Sentiment dataset: {len(SENTIMENT_DATA)} samples")
print(f"Vocabulary size: {sentiment_vocab_size}")
print(f"Input shape: {X.shape} | Labels shape: {y.shape}")

In [ ]:
# ──────────────────────────── 2. Classifier Architecture ─────────────────

class SentimentClassifier(nn.Module):
    """A simple sentiment classifier using embedding + mean-pooling + linear.

    Architecture:
        tokens → Embedding(V, D) → mean-pool over sequence → Linear(D, 1) → σ

    The critical parameter is `freeze_embeddings`:
        - True:  Embedding weights are NOT updated (frozen lookup table).
        - False: Embedding weights receive gradients and adapt to the task.
    """

    def __init__(
        self,
        vocab_size: int,
        embed_dim: int,
        freeze_embeddings: bool = False,
        pretrained_weights: torch.Tensor | None = None,
    ) -> None:
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        # Load pretrained weights if provided
        if pretrained_weights is not None:
            self.embedding.weight.data.copy_(pretrained_weights)

        # KEY DECISION: freeze or fine-tune
        self.embedding.weight.requires_grad = not freeze_embeddings

        self.classifier = nn.Linear(embed_dim, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass with mean-pooling over non-padding tokens.

        Args:
            x: Token indices of shape (B, L).

        Returns:
            Logits of shape (B,).
        """
        embedded: torch.Tensor = self.embedding(x)     # (B, L, D)

        # Mask out padding tokens for clean mean-pooling
        mask: torch.Tensor = (x != 0).unsqueeze(-1).float()  # (B, L, 1)
        # Sum embeddings where mask is 1, divide by count of non-pad tokens
        pooled: torch.Tensor = (embedded * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)

        logits: torch.Tensor = self.classifier(pooled).squeeze(-1)  # (B,)
        return logits

In [ ]:
# ──────────────────────────── 3. Training Harness ────────────────────────

def train_and_evaluate(
    model: nn.Module,
    X: torch.Tensor,
    y: torch.Tensor,
    n_epochs: int = 200,
    lr: float = 0.01,
    label: str = "Model",
) -> Dict[str, List[float]]:
    """Train a binary sentiment classifier and track loss + accuracy.

    Args:
        model:    The SentimentClassifier instance.
        X:        Input tensor of shape (N, L).
        y:        Binary labels of shape (N,).
        n_epochs: Number of training epochs.
        lr:       Learning rate.
        label:    Display name for logging.

    Returns:
        Dict with 'loss' and 'accuracy' histories.
    """
    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()), lr=lr
    )
    criterion = nn.BCEWithLogitsLoss()
    history: Dict[str, List[float]] = {"loss": [], "accuracy": []}

    for epoch in range(1, n_epochs + 1):
        model.train()
        optimizer.zero_grad()
        logits: torch.Tensor = model(X)
        loss: torch.Tensor = criterion(logits, y)
        loss.backward()
        optimizer.step()

        # Compute accuracy
        with torch.no_grad():
            preds: torch.Tensor = (torch.sigmoid(logits) > 0.5).float()
            accuracy: float = (preds == y).float().mean().item()

        history["loss"].append(loss.item())
        history["accuracy"].append(accuracy)

        if epoch % 50 == 0:
            print(f"  [{label}] Epoch {epoch:3d}/{n_epochs} | "
                  f"Loss: {loss.item():.4f} | Acc: {accuracy:.1%}")

    return history

In [ ]:
# ──────────────────────────── 4. Experiment: Frozen vs. Fine-Tuned ───────

SENTIMENT_EMBED_DIM: int = 32
SENTIMENT_EPOCHS: int = 200

# Create shared pretrained weights (simulating pre-trained Word2Vec)
# We use the same random initialization for a fair comparison.
torch.manual_seed(42)
pretrained_weights: torch.Tensor = torch.randn(sentiment_vocab_size, SENTIMENT_EMBED_DIM) * 0.1

print("=" * 65)
print("EXPERIMENT: Frozen vs. Fine-Tuned Embeddings")
print("=" * 65)

# ── A. Frozen Embeddings ──
print("\n── A. FROZEN Embeddings (requires_grad=False) ──")
frozen_model = SentimentClassifier(
    vocab_size=sentiment_vocab_size,
    embed_dim=SENTIMENT_EMBED_DIM,
    freeze_embeddings=True,
    pretrained_weights=pretrained_weights.clone(),
)
# Verify: only the linear classifier has trainable params
frozen_params = sum(p.numel() for p in frozen_model.parameters() if p.requires_grad)
print(f"  Trainable parameters: {frozen_params}")
frozen_history = train_and_evaluate(
    frozen_model, X, y, n_epochs=SENTIMENT_EPOCHS, label="Frozen"
)

# ── B. Fine-Tuned Embeddings ──
print("\n── B. FINE-TUNED Embeddings (requires_grad=True) ──")
finetuned_model = SentimentClassifier(
    vocab_size=sentiment_vocab_size,
    embed_dim=SENTIMENT_EMBED_DIM,
    freeze_embeddings=False,
    pretrained_weights=pretrained_weights.clone(),
)
finetuned_params = sum(p.numel() for p in finetuned_model.parameters() if p.requires_grad)
print(f"  Trainable parameters: {finetuned_params}")
finetuned_history = train_and_evaluate(
    finetuned_model, X, y, n_epochs=SENTIMENT_EPOCHS, label="Fine-Tuned"
)

In [ ]:
# ──────────────────────────── 5. Visualization ───────────────────────────

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Loss Curves ──
axes[0].plot(frozen_history["loss"], label="Frozen", linewidth=2, color="#e74c3c")
axes[0].plot(finetuned_history["loss"], label="Fine-Tuned", linewidth=2, color="#2ecc71")
axes[0].set_xlabel("Epoch", fontsize=12)
axes[0].set_ylabel("BCE Loss", fontsize=12)
axes[0].set_title("Training Loss: Frozen vs. Fine-Tuned", fontsize=14, fontweight="bold")
axes[0].legend(fontsize=12)
axes[0].grid(True, alpha=0.3)

# ── Accuracy Curves ──
axes[1].plot(frozen_history["accuracy"], label="Frozen", linewidth=2, color="#e74c3c")
axes[1].plot(finetuned_history["accuracy"], label="Fine-Tuned", linewidth=2, color="#2ecc71")
axes[1].set_xlabel("Epoch", fontsize=12)
axes[1].set_ylabel("Accuracy", fontsize=12)
axes[1].set_title("Training Accuracy: Frozen vs. Fine-Tuned", fontsize=14, fontweight="bold")
axes[1].legend(fontsize=12)
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

# ── Summary ──
print("\n" + "=" * 65)
print("RESULTS SUMMARY")
print("=" * 65)
print(f"  Frozen     → Final Acc: {frozen_history['accuracy'][-1]:.1%} | "
      f"Final Loss: {frozen_history['loss'][-1]:.4f}")
print(f"  Fine-Tuned → Final Acc: {finetuned_history['accuracy'][-1]:.1%} | "
      f"Final Loss: {finetuned_history['loss'][-1]:.4f}")
print("\n📌 Interpretation:")
print("  Fine-tuned embeddings converge faster and reach higher accuracy")
print("  because they can adapt the vector space to separate sentiment-")
print("  bearing words (e.g., 'amazing' far from 'terrible') in ways that")
print("  generic pre-trained embeddings cannot.")

### Embedding Space Analysis

Let's visualize **how fine-tuning deforms the embedding space** to make it task-specific.

In [ ]:
# ──────────────────────────── 6. Embedding Drift Analysis ────────────────
from sklearn.metrics.pairwise import cosine_similarity

# Select sentiment-critical words to analyze
PROBE_WORDS: List[str] = ["amazing", "terrible", "great", "awful", "good", "bad"]

print("Cosine Similarity: Pre-trained (Frozen) vs. Fine-Tuned Embeddings")
print("=" * 65)

for w1, w2 in [("amazing", "terrible"), ("amazing", "great"), ("bad", "awful")]:
    idx1: int = sentiment_vocab[w1]
    idx2: int = sentiment_vocab[w2]

    # Frozen model embeddings (≈ original pretrained)
    v1_frozen: np.ndarray = frozen_model.embedding.weight[idx1].detach().numpy()
    v2_frozen: np.ndarray = frozen_model.embedding.weight[idx2].detach().numpy()
    sim_frozen: float = float(cosine_similarity([v1_frozen], [v2_frozen])[0][0])

    # Fine-tuned model embeddings
    v1_ft: np.ndarray = finetuned_model.embedding.weight[idx1].detach().numpy()
    v2_ft: np.ndarray = finetuned_model.embedding.weight[idx2].detach().numpy()
    sim_ft: float = float(cosine_similarity([v1_ft], [v2_ft])[0][0])

    drift: float = sim_ft - sim_frozen
    direction: str = "↑ closer" if drift > 0 else "↓ farther"

    print(f"  ({w1:>9s}, {w2:<9s}) | Frozen: {sim_frozen:+.3f} | "
          f"Fine-Tuned: {sim_ft:+.3f} | Drift: {drift:+.3f} {direction}")

print("\n📌 Key Insight:")
print("  After fine-tuning, same-sentiment words (amazing↔great) move CLOSER,")
print("  while opposite-sentiment words (amazing↔terrible) move FARTHER apart.")
print("  This is the embedding space 'deforming' to serve the classification task.")

--- 
## ✅ Knowledge Check

### Q1: The Softmax Bottleneck
Why does the standard Word2Vec objective function require Negative Sampling or Hierarchical Softmax for production use?

<details><summary>👉 Answer</summary>
Standard Softmax requires calculating the dot product for EVERY word in the vocabulary for every training step. With a vocab of 500k+, this is computationally prohibitive. Negative Sampling simplifies this into a binary task against a handful of noise words, reducing complexity from O(V) to O(k), where k is the number of negative samples (usually 5-20). Hierarchical Softmax reduces it to O(log V) by replacing the flat output layer with a binary tree of binary classifiers.
</details>

### Q2: Hierarchical Softmax Probability Guarantee
In Hierarchical Softmax, why does $\sum_{w=1}^V P(w | v_c) = 1$ hold automatically, and why does Negative Sampling NOT provide this guarantee?

<details><summary>👉 Answer</summary>
In Hierarchical Softmax, each internal node routes probability mass left or right: $P(\text{left}) + P(\text{right}) = 1$. Since every leaf (word) is reached by a unique path, the total probability over all leaves sums to 1 by construction. Negative Sampling, by contrast, trains independent binary classifiers for each word pair — there is no normalization mechanism ensuring the scores across all words sum to 1. In practice this doesn't matter because we only need relative rankings, not calibrated probabilities.
</details>

### Q3: FastText vs. Word2Vec
In a production environment for a highly technical domain (e.g., Medicine or Chemistry) where many rare/unseen words appear, which embedding model should you choose?

<details><summary>👉 Answer</summary>
FastText. Technical domains often have complex morphology (e.g., 'dihydrogen monoxide'). FastText's subword information allows it to generalize to unseen chemical or medical terms by leveraging prefixes and suffixes used in known words.
</details>

### Q4: Frozen vs. Fine-Tuned Embeddings
Under what conditions would you prefer frozen embeddings over fine-tuned, even if fine-tuning gives higher accuracy?

<details><summary>👉 Answer</summary>
1. **Very small labeled dataset**: Fine-tuning a large embedding matrix with few labels risks overfitting — the embeddings memorize the training set rather than learning generalizable sentiment.
2. **Multi-task deployment**: If the same embeddings are used across many downstream tasks, fine-tuning them for one task degrades performance on others.
3. **Interpretability**: Frozen embeddings preserve the original semantic structure (analogies, similarity), which may be needed for explainability.
4. **Training budget**: Freezing reduces the number of trainable parameters dramatically, speeding up training and reducing memory.
</details>

### Q5: When to Use Static Embeddings?
If Transformers (BERT/GPT) provide superior contextual embeddings, why would an engineer ever use static embeddings like Word2Vec today?

<details><summary>👉 Answer</summary>
1. **Inference Speed**: Table lookups for static embeddings are nearly instantaneous. Transformers require massive GPU compute.
2. **Dimensionality**: Word2Vec (100-300D) is much smaller than modern embeddings (768-4096D), making it cheaper for large-scale storage/similarity search.
3. **Baselines**: For simple classification tasks, Word2Vec + a Linear layer is a powerful baseline that avoids the overhead of LLMs.
</details>

--- 
## 🔨 Practical Practice

1. **Modify the SGNS Model**: Add a learning rate scheduler and observe the effect on loss convergence.
2. **The Analogy Explorer**: Find a word pair in the `gensim` pre-trained models (e.g., `glove-wiki-gigaword-100`) that demonstrates a geographical analogy (e.g., `Paris - France + Germany = Berlin`).
3. **Bias Analysis**: Calculate the similarity between `doctor` and `man` vs `doctor` and `woman` in a pre-trained Word2Vec model. Reflect on how historical data bias manifests in the representation space.
4. **Extend the Extrinsic Experiment**: Replace the random pretrained weights with actual `gensim` Word2Vec embeddings and compare frozen vs. fine-tuned accuracy on a larger sentiment dataset.

**Next →** `16_03_recurrent_networks_and_lstms.ipynb` — NLP Masterclass 03: Recurrent Networks, LSTMs & 1D CNNs for Text

## 12 · Embedding Bias, Fairness, and WEAT

As representation learning captures the statistical regularities of human language from large-scale corpora, static embeddings invariably inherit, and can even amplify, historical human biases. For instance:
- **Gender Bias:** Models often associate "programmer" closer to male pronouns and "nurse" closer to female pronouns.
- **Racial/Ethnic Bias:** Names associated with certain ethnicities might be embedded closer to negative sentiment words than names from other ethnicities.
- **Occupational Bias:** Certain jobs are correlated stereotypically with different demographic groups.

These vectors encode correlation, not necessarily ground-truth semantic definition. Because the embeddings map words to a continuous vector space where distance equals semantic similarity, sociocognitive biases crystallize into pure geometric distance.

### The Word Embedding Association Test (WEAT)

To mathematically measure this geometric bias, researchers introduced **WEAT** (Caliskan et al., 2017), an adaptation of the psychological Implicit Association Test (IAT) for word embeddings. WEAT measures the relative cosine similarity between two sets of *Target Words* (e.g., Male vs. Female names) and two sets of *Attribute Words* (e.g., Career vs. Family words).

Let $X$ and $Y$ be the two sets of target words (e.g., Male terms $X$, Female terms $Y$).
Let $A$ and $B$ be the two sets of attribute words (e.g., Career words $A$, Family words $B$).

For a given single word $w$, its association with the attributes is the difference in mean cosine similarity:
$$ s(w, A, B) = \text{mean}_{a \in A} \cos(w, a) - \text{mean}_{b \in B} \cos(w, b) $$

The overall WEAT score for the sets is the sum of these associations over $X$ minus the sum over $Y$:
$$ s(X, Y, A, B) = \sum_{x \in X} s(x, A, B) - \sum_{y \in Y} s(y, A, B) $$

A positive score implies that $X$ is more associated with $A$ and $Y$ is more associated with $B$. To determine significance, an effect size is calculated by dividing by the standard deviation of associations across all words in $X \cup Y$.

In [ ]:
import numpy as np

# Let's create some dummy embeddings to simulate a biased vector space
# In a real scenario, these would be vectors extracted from a trained Word2Vec or GloVe model.
np.random.seed(42)

# Dummy vocabulary and vectors (engineered for demonstration)
vocab = {
    # Targets (X: Male terms, Y: Female terms)
    "he": np.array([0.8, 0.2]), "him": np.array([0.7, 0.3]), "man": np.array([0.9, 0.1]),
    "she": np.array([0.2, 0.8]), "her": np.array([0.3, 0.7]), "woman": np.array([0.1, 0.9]),
    # Attributes (A: Career words, B: Family words)
    "executive": np.array([0.9, 0.3]), "management": np.array([0.8, 0.2]), "professional": np.array([0.7, 0.4]),
    "home": np.array([0.2, 0.9]), "parents": np.array([0.3, 0.8]), "children": np.array([0.1, 1.0])
}

# Define sets
X = ["he", "him", "man"]
Y = ["she", "her", "woman"]
A = ["executive", "management", "professional"]
B = ["home", "parents", "children"]

def cosine_similarity(v1, v2):
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

def word_association(w, A_set, B_set, embeddings):
    v_w = embeddings[w]
    mean_A = np.mean([cosine_similarity(v_w, embeddings[a]) for a in A_set])
    mean_B = np.mean([cosine_similarity(v_w, embeddings[b]) for b in B_set])
    return mean_A - mean_B

def weat_score(X_set, Y_set, A_set, B_set, embeddings):
    score_X = np.sum([word_association(x, A_set, B_set, embeddings) for x in X_set])
    score_Y = np.sum([word_association(y, A_set, B_set, embeddings) for y in Y_set])
    return score_X - score_Y

print("--- Measuring Geometric Bias with Simplified WEAT ---")
association_man = word_association("man", A, B, vocab)
print(f"Association for 'man' (Career vs Family): {association_man:.4f}")

association_woman = word_association("woman", A, B, vocab)
print(f"Association for 'woman' (Career vs Family): {association_woman:.4f}")

total_weat = weat_score(X, Y, A, B, vocab)
print(f"\nTotal WEAT Score (X&A vs Y&B): {total_weat:.4f}")
print("A positive WEAT score indicates 'Male' words align more with 'Career'")
print("and 'Female' words align more with 'Family' in this embedding space.")

### Mitigation: Hard Debiasing

While measuring bias is the critical first step, mitigating it has become a central focus of NLP research. One prominent post-processing technique is **Hard Debiasing** (Bolukbasi et al., 2016).

The process generally involves:
1. **Identify the Bias Direction:** Collect a set of definitionally gendered word pairs (e.g., *he-she, man-woman*) and compute their vector differences. Perform Principal Component Analysis (PCA) on these difference vectors to identify the subspace (often a single principal component) that most strongly captures the "gender" direction.
2. **Neutralize:** For gender-neutral words (e.g., *programmer, nurse, doctor*), project their vectors orthogonally away from the gender direction. This mathematically forces them to have zero projection along the bias axis, effectively removing the geometric correlation with gender.
3. **Equalize:** Ensure that pairs of gendered words are equidistant from neutral words.

> **Critical Insight:** While Hard Debiasing forcefully zeroes out the targeted linear bias, later research (e.g., Gonen & Goldberg, 2019) demonstrated that biases are insidious. Non-linear proxies and secondary associations often remain encoded in the embedding geometry. Because of this, modern approaches often pivot towards jointly learning fair embeddings during the initial pretraining objective or applying sophisticated techniques directly within the attention heads of Transformer models.